# Hansen Ch.14 Time Series — 计算

理论见 `Hansen_Ch14_Exercises_Solutions.md`（**14.1–14.22**）。

实证：**14.18–14.22**（FRED-QD / FRED-MD）。


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import inv
from scipy import stats

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")
qd = pd.read_excel(ROOT / "FRED-QD/FRED-QD.xlsx")
md = pd.read_excel(ROOT / "FRED-MD/FRED-MD.xlsx")

def ols_hc(y, X):
    n, k = X.shape
    b = inv(X.T @ X) @ (X.T @ y)
    e = y - X @ b
    u = X * e[:, None]
    V = inv(X.T @ X) @ (u.T @ u) @ inv(X.T @ X)
    return b, e, V, n, k

def newey_west(X, e, M=5):
    n, k = X.shape
    u = X * e[:, None]
    S = u.T @ u
    for j in range(1, M + 1):
        w = 1 - j / (M + 1)
        Gj = u[j:].T @ u[:-j]
        S += w * (Gj + Gj.T)
    return inv(X.T @ X) @ S @ inv(X.T @ X)

def ar_xy(y, p):
    y = np.asarray(y, float)
    Y = y[p:]
    X = np.column_stack([y[p - j : len(y) - j] for j in range(1, p + 1)] + [np.ones(len(y) - p)])
    return Y, X

def wald(R, b, V):
    R = np.atleast_2d(R)
    d = R @ b
    W = float(d.T @ inv(R @ V @ R.T) @ d)
    return W, float(1 - stats.chi2.cdf(W, R.shape[0]))


## 14.18 `pnfix` 季度增长率 AR(4)

In [ ]:

pnfi = qd["pnfix"].astype(float)
g = 100 * (pnfi / pnfi.shift(1) - 1).dropna().to_numpy()
Y, X = ar_xy(g, 4)
b, e, Vhc, n, k = ols_hc(Y, X)
Vnw = newey_west(X, e, 5)
tab = pd.DataFrame({
    "coef": b,
    "SE_HC": np.sqrt(np.diag(Vhc)),
    "SE_NW_M5": np.sqrt(np.diag(Vnw)),
}, index=[f"lag{i}" for i in range(1, 5)] + ["const"])
print("n =", n)
print(tab)
# IRF
A = np.zeros((4, 4))
A[0, :] = b[:4]
A[1, 0] = A[2, 1] = A[3, 2] = 1.0
x = np.array([1.0, 0, 0, 0])
irf = []
for j in range(10):
    x = A @ x
    irf.append(x[0])
print("IRF j=1..10:", np.round(irf, 4))


## 14.19 `oilpricex` 一阶差分 AR(4) + 随机游走检验

In [ ]:

oil = qd["oilpricex"].astype(float)
doil = oil.diff().dropna().to_numpy()
Y, X = ar_xy(doil, 4)
b, e, V, n, k = ols_hc(Y, X)
print("n =", n)
print(pd.Series(b, index=[f"a{i}" for i in range(1,5)]+["const"]))
R = np.zeros((4, 5))
for i in range(4):
    R[i, i] = 1
W, p = wald(R, b, V)
print(f"Wald H0: AR coeffs=0: W={W:.3f}, p={p:.4f}")


## 14.20 `unrate` 月度 AR(1)–AR(8)，1960m1 起同一样本

In [ ]:

un = md["unrate"].astype(float).to_numpy()
start = 12  # 1960m1 if series starts 1959m1
rows = []
for p in range(1, 9):
    Y = un[start:]
    nT = len(Y)
    X = np.column_stack([un[start - j : start - j + nT] for j in range(1, p + 1)] + [np.ones(nT)])
    b, e, V, _, _ = ols_hc(Y, X)
    s2 = (e @ e) / nT
    aic = np.log(s2) + 2 * (p + 1) / nT
    rows.append({"p": p, "AIC": aic, "n": nT})
    if p == min(range(1,9), key=lambda pp: rows[pp-1]["AIC"] if pp<=len(rows) else 1e9):
        pass
aic_df = pd.DataFrame(rows)
print(aic_df.to_string(index=False))
pstar = int(aic_df.loc[aic_df.AIC.idxmin(), "p"])
print("Selected p =", pstar)
Y = un[start:]
nT = len(Y)
X = np.column_stack([un[start - j : start - j + nT] for j in range(1, pstar + 1)] + [np.ones(nT)])
b, e, V, _, _ = ols_hc(Y, X)
print(pd.DataFrame({"coef": b, "SE_HC": np.sqrt(np.diag(V))},
                   index=[f"lag{i}" for i in range(1, pstar+1)] + ["const"]))


## 14.21 失业率与 claimsx；14.22 GDP 增长与 houst

In [ ]:

def lagmat(y, p):
    n = len(y)
    return np.column_stack([y[p - j : n - j] for j in range(1, p + 1)])

# 14.21
df = pd.DataFrame({"un": qd["unrate"].astype(float), "cl": qd["claimsx"].astype(float)}).dropna()
un, cl = df.un.values, df.cl.values
p = 4
Y = un[p:]
X = np.column_stack([lagmat(cl, p), np.ones(len(Y))])
b, e, V, n, k = ols_hc(Y, X)
print("14.21 DL claims -> unrate n=", n)
print("coef", b)
X2 = np.column_stack([lagmat(un, p), lagmat(cl, p), np.ones(len(Y))])
b2, e2, V2, _, _ = ols_hc(Y, X2)
R = np.zeros((4, 9))
for i in range(4):
    R[i, 4 + i] = 1
print("ADL Granger claims->un:", wald(R, b2, V2))

# 14.22
gdp = qd["gdpc1"].astype(float)
g = 100 * (gdp / gdp.shift(1) - 1)
df = pd.DataFrame({"g": g, "h": qd["houst"].astype(float)}).dropna()
g, h = df.g.values, df.h.values
Y = g[4:]
X = np.column_stack([g[3:-1], g[2:-2], lagmat(h, 4), np.ones(len(Y))])
b, e, V, n, k = ols_hc(Y, X)
print("\n14.22 ADL GDP growth <- houst n=", n)
print("coef", b)
R = np.zeros((4, 7))
for i in range(4):
    R[i, 2 + i] = 1
print("Granger houst->g:", wald(R, b, V))
